# 04 — Simulação de fila operacional de alertas

Este notebook converte os alertas do baseline em um cenário reproduzível de triagem. **Todos os horários, equipe, SLA e duração de análise são premissas de simulação**, não dados observados de uma operação. O PaySim não registra chegadas intrastep, analistas, decisões de triagem ou pendências reais.

A simulação principal usa a partição de validação e o limiar 2, sem usar o teste exploratório para escolher a capacidade. Os rótulos PaySim são usados separadamente apenas para calcular precisão/recall retrospectivos; não são exportados para a fila operacional simulada. Os resultados não estimam perdas evitadas nem desempenho real.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'src').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.data.prepare_data import find_csv
from src.analysis.risk_rules import chronological_masks, fit_rule_thresholds, score_rules, alert_metrics, average_precision_stepwise
from src.analysis.queue_simulation import simulate_queue
csv_path = find_csv(ROOT / 'data' / 'raw')
df = pd.read_csv(csv_path)
train_mask, valid_mask, test_mask = chronological_masks(df)
p95_by_type = fit_rule_thresholds(df.loc[train_mask])
df['risk_score_baseline'] = score_rules(df, p95_by_type)
valid = df.loc[valid_mask].copy()
print(f'Base carregada: {len(df):,} transações; validação: {len(valid):,}.')

## Premissas do cenário-base

| Elemento | Premissa de demonstração |
|---|---|
| Unidade temporal | 1 `step` é mapeado para 60 minutos; chegada uniforme dentro do intervalo, pois não há horário intrastep no arquivo usado |
| Equipe | 5 analistas simulados, disponíveis durante todo o horizonte |
| Prioridade | Score 3 = P1; score 2 = P2; ordenar por prioridade e depois chegada |
| Tempo de análise | Distribuição triangular por score, em minutos; parâmetros ilustrativos e editáveis abaixo |
| SLA | Tempo máximo até início da triagem: P1 15 min, P2 30 min |
| Reprodutibilidade | Semente fixa 2026 |

A distribuição triangular é uma hipótese de capacidade, não uma medição. Teste diferentes valores antes de apresentar uma conclusão; não interprete esse exercício como SLA observado.


In [ ]:
THRESHOLD = 2
POLICY = {
    'analysts': 5,
    'minutes_per_step': 60,
    'sla_minutes': {3: 15, 2: 30, 1: 60, 0: 60},
    'service_triangular_minutes': {
        3: (6.0, 10.0, 16.0),
        2: (4.0, 7.0, 12.0),
        1: (3.0, 5.0, 9.0),
        0: (3.0, 5.0, 9.0),
    },
}
queue, operational_kpis, retrospective_metrics = simulate_queue(
    valid, threshold=THRESHOLD, policy=POLICY, seed=2026
)
display(pd.Series(operational_kpis, name='Estimativa operacional simulada'))
display(pd.Series(retrospective_metrics, name='Backtest retrospectivo PaySim'))


## Interpretação dos indicadores

- **Espera até triagem** e **pendências** são estimativas do cenário de equipe/tempos acima.
- **Dentro do SLA** mede a proporção dos alertas iniciados dentro da meta assumida; pendentes fora do SLA são contados no fim do horizonte simulado.
- **Precisão e recall** comparam o alerta retrospectivamente ao rótulo `isFraud` da validação. Não são indicadores operacionais nem validação independente.
- A utilização é uma razão aproximada entre minutos de serviço simulados e minutos de capacidade nominal; interpreta-se com cautela devido às chegadas e ao horizonte finito.
- Os identificadores `SIM-...` são artificiais. O arquivo da fila não inclui nome, conta, rótulo de fraude ou identificador PaySim.


In [ ]:
# Sensibilidade simples: variar equipe mantendo as demais premissas.
sensitivity = []
for analyst_count in (2, 5, 10):
    policy = {**POLICY, 'analysts': analyst_count}
    _, ops, backtest = simulate_queue(valid, threshold=THRESHOLD, policy=policy, seed=2026)
    sensitivity.append({'analistas': analyst_count, **ops, 'precisao_backtest_pct': backtest['precisao_pct'], 'recall_backtest_pct': backtest['recall_pct']})
sensitivity_df = pd.DataFrame(sensitivity)
display(sensitivity_df[['analistas', 'alertas', 'triados_ate_fim', 'concluidos_ate_fim', 'pendentes_na_fila_ao_fim', 'espera_media_min', 'espera_p90_min', 'triados_dentro_sla_pct', 'utilizacao_estimada_pct', 'precisao_backtest_pct', 'recall_backtest_pct']])

In [ ]:
# Revisão de uma amostra sem rótulos nem identificadores originais.
display(queue[['alert_id', 'arrival_step', 'priority', 'score', 'arrival_minute', 'triage_start_minute', 'wait_minutes', 'sla_minutes', 'service_minutes', 'status_at_horizon']].head(20))

# Exporta resultados sintéticos somente após execução local com a base PaySim.
processed_dir = ROOT / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)
queue.to_csv(processed_dir / 'simulated_alert_queue_validation.csv', index=False)
pd.DataFrame([operational_kpis]).to_csv(processed_dir / 'simulated_queue_kpis_validation.csv', index=False)
sensitivity_df.to_csv(processed_dir / 'simulated_capacity_sensitivity_validation.csv', index=False)
print('Resultados simulados exportados. Identificadores artificiais; não são registros operacionais.')

## Limites e próximos dados necessários

Para substituir estimativas por KPIs reais seriam necessários logs com, no mínimo, horário de criação do alerta, início da análise, decisão/encaminhamento, encerramento, prioridade, equipe/turno e estado pendente em cada corte. A base PaySim não contém isso. Uma implementação real ainda exigiria validação de regras de negócio, privacidade, controles de acesso, calibração de carga e revisão humana.

**Conclusão permitida:** comparar cenários hipotéticos de capacidade para o portfólio. **Conclusão não permitida:** afirmar que a equipe real atenderia esses SLAs, que os pendentes ocorreram de verdade, ou que houve fraude/prejuízo evitado em operação.